# Sentinel-2 NDVI Processing

This notebook continues the GeoSoil-Cuba Sentinel-2 workflow after `03_sentinel2_data_acquisition.ipynb`. It uses the validated Sentinel-2 B04 (red) and B08 (near-infrared) acquisition pairs to calculate tile-level NDVI, create AOI-level NDVI rasters, validate the outputs, and select additional acquisition dates for the planned time series.

## Workflow

1. Calculate NDVI for each complete B04/B08 tile pair.
2. Mosaic the tile-level NDVI rasters and crop them to the Jaruco/Mayabeque area of interest (AOI).
3. Validate the AOI-level rasters and select additional dates to improve temporal coverage.
4. Export available tables and figures for reporting.

## Inputs and outputs

- **Input imagery:** validated B04 and B08 JP2 files under `data/raw/satellite/sentinel2/`.
- **AOI vector:** `data/processed/vector/soil_vegetation_analysis.gpkg`.
- **Tile-level NDVI:** `data/processed/satellite/ndvi/`.
- **AOI-level NDVI:** `data/processed/satellite/ndvi_aoi/`.
- **Report tables and figures:** `outputs/tables/` and `outputs/figures/`.

The workflow reuses existing NDVI GeoTIFFs where possible. Run the cells in order because later steps depend on DataFrames and geometry objects created earlier in the notebook. The project root is assumed to be the parent directory of the notebook folder.


In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio


# ---------------------------------------------------------
# 1. Project paths
# ---------------------------------------------------------

PROJECT_DIR = Path("..").resolve()

RAW_SATELLITE_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "satellite"
    / "sentinel2"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_SATELLITE_DIR = (
    PROCESSED_DIR
    / "satellite"
)

SOIL_VEGETATION_ANALYSIS_GPKG = (
    PROCESSED_DIR
    / "vector"
    / "soil_vegetation_analysis.gpkg"
)


# ---------------------------------------------------------
# 2. Verify required project inputs
# ---------------------------------------------------------

print(f"Project root: {PROJECT_DIR}")
print(f"Raw Sentinel-2 directory exists: {RAW_SATELLITE_DIR.is_dir()}")
print(
    "Soil–vegetation GeoPackage exists: "
    f"{SOIL_VEGETATION_ANALYSIS_GPKG.is_file()}"
)

if not RAW_SATELLITE_DIR.is_dir():
    raise FileNotFoundError(
        f"Sentinel-2 input directory not found: {RAW_SATELLITE_DIR}"
    )

if not SOIL_VEGETATION_ANALYSIS_GPKG.is_file():
    raise FileNotFoundError(
        "Required AOI GeoPackage not found: "
        f"{SOIL_VEGETATION_ANALYSIS_GPKG}"
    )


## 1. Calculate Tile-Level NDVI

For each complete acquisition, NDVI is calculated independently for T17QLF and T17QMF.

The Sentinel-2 L2A reflectance values use a scale factor of `0.0001` and an offset of `-0.1`. Because the same scale and offset apply to B04 and B08, the offset cancels mathematically in the normalized difference when applied consistently to both bands. We therefore calculate NDVI directly from the stored reflectance values.


In [ ]:
## Generate NDVI rasters only for missing outputs
##
## Existing NDVI GeoTIFFs are reused and are not regenerated.


import numpy as np
import pandas as pd
import rasterio
from pathlib import Path


# ---------------------------------------------------------
# 1. Output directory
# ---------------------------------------------------------

ndvi_dir = PROCESSED_SATELLITE_DIR / "ndvi"

ndvi_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# 2. Process each validated B04/B08 pair
# ---------------------------------------------------------

ndvi_records = []

grouped = downloaded_bands.groupby(
    [
        "acquisition_date",
        "satellite",
        "tile"
    ]
)


for (
    acquisition_date,
    satellite,
    tile
), group in grouped:

    # Locate B04 and B08
    b04_row = group[
        group["band"] == "B04"
    ]

    b08_row = group[
        group["band"] == "B08"
    ]

    # Skip incomplete pairs
    if len(b04_row) != 1 or len(b08_row) != 1:

        print(
            f"Skipping incomplete pair: "
            f"{acquisition_date.date()} "
            f"{satellite} "
            f"{tile}"
        )

        continue


    b04_path = b04_row.iloc[0]["path"]
    b08_path = b08_row.iloc[0]["path"]

    output_path = (
        ndvi_dir
        / f"{acquisition_date:%Y-%m-%d}_"
          f"{satellite}_"
          f"{tile}_NDVI.tif"
    )


    # -----------------------------------------------------
    # 3. Reuse existing NDVI raster
    # -----------------------------------------------------

    if output_path.exists():

        print(
            f"Already exists — skipping: "
            f"{output_path.name}"
        )

        with rasterio.open(output_path) as src:

            ndvi = src.read(1)

            valid = ndvi != src.nodata

            valid_pixels = np.count_nonzero(valid)

            ndvi_records.append(
                {
                    "acquisition_date": acquisition_date,
                    "satellite": satellite,
                    "tile": tile,
                    "path": str(output_path),
                    "valid_pixels": valid_pixels,
                    "ndvi_min": (
                        float(ndvi[valid].min())
                        if valid_pixels > 0
                        else np.nan
                    ),
                    "ndvi_max": (
                        float(ndvi[valid].max())
                        if valid_pixels > 0
                        else np.nan
                    )
                }
            )

        continue


    print(
        f"Creating: "
        f"{output_path.name}"
    )


    # -----------------------------------------------------
    # 4. Open B04 and B08
    # -----------------------------------------------------

    with rasterio.open(b04_path) as b04_src, \
         rasterio.open(b08_path) as b08_src:

        # Check spatial compatibility
        if (
            b04_src.crs != b08_src.crs
            or b04_src.transform != b08_src.transform
            or b04_src.width != b08_src.width
            or b04_src.height != b08_src.height
        ):

            raise ValueError(
                f"B04/B08 geometry mismatch: "
                f"{acquisition_date.date()} "
                f"{satellite} {tile}"
            )


        b04 = b04_src.read(1).astype(
            np.float32
        )

        b08 = b08_src.read(1).astype(
            np.float32
        )

        b04_nodata = b04_src.nodata
        b08_nodata = b08_src.nodata

        profile = b04_src.profile.copy()


        # -------------------------------------------------
        # 5. Build valid-data mask
        # -------------------------------------------------

        valid = np.isfinite(b04) & np.isfinite(b08)

        if b04_nodata is not None:
            valid &= b04 != b04_nodata

        if b08_nodata is not None:
            valid &= b08 != b08_nodata


        # -------------------------------------------------
        # 6. Calculate NDVI
        # -------------------------------------------------

        ndvi = np.full(
            b04.shape,
            -9999.0,
            dtype=np.float32
        )

        denominator = b08 + b04

        valid &= denominator != 0

        ndvi[valid] = (
            (b08[valid] - b04[valid])
            / denominator[valid]
        )


        # -------------------------------------------------
        # 7. Write NDVI GeoTIFF
        # -------------------------------------------------

        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            nodata=-9999.0,
            compress="deflate"
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(
                ndvi,
                1
            )

            dst.set_band_description(
                1,
                "NDVI"
            )


    # -----------------------------------------------------
    # 8. Record newly created result
    # -----------------------------------------------------

    valid_pixels = np.count_nonzero(
        ndvi != -9999.0
    )

    ndvi_records.append(
        {
            "acquisition_date": acquisition_date,
            "satellite": satellite,
            "tile": tile,
            "path": str(output_path),
            "valid_pixels": valid_pixels,
            "ndvi_min": (
                float(ndvi[ndvi != -9999.0].min())
                if valid_pixels > 0
                else np.nan
            ),
            "ndvi_max": (
                float(ndvi[ndvi != -9999.0].max())
                if valid_pixels > 0
                else np.nan
            )
        }
    )


# ---------------------------------------------------------
# 9. Create NDVI inventory
# ---------------------------------------------------------

ndvi_inventory = pd.DataFrame(
    ndvi_records
)

print(
    f"\nNDVI rasters available: "
    f"{len(ndvi_inventory)}"
)

display(
    ndvi_inventory
)

## 2. Create AOI-Level NDVI Rasters

The `soil_vegetation_analysis.gpkg` is the authoritative vector source for the project AOI.

The vector data use EPSG:3795, while the Sentinel-2 NDVI rasters use EPSG:32617. Therefore, the AOI geometry is reprojected to EPSG:32617 before raster masking.


In [ ]:
import geopandas as gpd


# ---------------------------------------------------------
# 1. Load the project AOI source
# ---------------------------------------------------------

aoi_path = (
    PROJECT_DIR / "data" / "processed" / "vector" / "soil_vegetation_analysis.gpkg"
)


# ---------------------------------------------------------
# 2. Read the soil–vegetation analysis layer
# ---------------------------------------------------------

soil_vegetation = gpd.read_file(
    aoi_path,
    layer="soil_vegetation"
)


# ---------------------------------------------------------
# 3. Confirm the vector CRS
# ---------------------------------------------------------

print(
    "Vector CRS:",
    soil_vegetation.crs
)


# ---------------------------------------------------------
# 4. Create the AOI geometry
# ---------------------------------------------------------
#
# The soil–vegetation layer covers the analysis AOI.
# Dissolving creates one geometry for masking the raster.

aoi = soil_vegetation.dissolve()


# ---------------------------------------------------------
# 5. Reproject AOI to Sentinel-2 raster CRS
# ---------------------------------------------------------

aoi_raster_crs = "EPSG:32617"

aoi_raster = aoi.to_crs(
    aoi_raster_crs
)


# ---------------------------------------------------------
# 6. Convert to the geometry format expected by rasterio
# ---------------------------------------------------------

aoi_geometry = [
    geometry
    for geometry in aoi_raster.geometry
]


print(
    "Raster CRS:",
    aoi_raster.crs
)

print(
    "AOI geometry count:",
    len(aoi_geometry)
)


### 2.1 Mosaic and Crop NDVI Rasters to the AOI

The two Sentinel-2 tiles covering the study area are mosaicked for each acquisition date. The resulting mosaic is then cropped to the Jaruco AOI.


In [ ]:
import numpy as np
import pandas as pd
import rasterio

from rasterio.merge import merge
from rasterio.mask import mask


# ---------------------------------------------------------
# 1. Output directory
# ---------------------------------------------------------

ndvi_aoi_dir = PROCESSED_SATELLITE_DIR / "ndvi_aoi"

ndvi_aoi_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# 2. Process each acquisition
# ---------------------------------------------------------

ndvi_aoi_records = []

for acquisition_date in sorted(
    ndvi_inventory["acquisition_date"].unique()
):

    date_rows = ndvi_inventory[
        ndvi_inventory["acquisition_date"]
        == acquisition_date
    ]

    # There should be exactly two tiles
    if len(date_rows) != 2:

        print(
            f"Skipping {acquisition_date:%Y-%m-%d}: "
            f"expected 2 tiles, found {len(date_rows)}"
        )

        continue

    satellite = date_rows.iloc[0]["satellite"]

    # -----------------------------------------------------
    # 3. Define output path before processing
    # -----------------------------------------------------

    output_path = (
        ndvi_aoi_dir
        / f"{acquisition_date:%Y-%m-%d}_"
          f"{satellite}_NDVI_AOI.tif"
    )


    # -----------------------------------------------------
    # 4. Skip existing AOI raster
    # -----------------------------------------------------

    if output_path.exists():

        print(
            f"Already exists, skipping: "
            f"{output_path.name}"
        )

        # Read existing raster statistics so that the
        # inventory contains both old and new rasters.
        with rasterio.open(output_path) as src:

            values = src.read(1)

            valid = values != src.nodata
            valid_values = values[valid]

            ndvi_aoi_records.append(
                {
                    "acquisition_date": acquisition_date,
                    "satellite": satellite,
                    "path": str(output_path),
                    "width": src.width,
                    "height": src.height,
                    "valid_pixels": int(valid.sum()),
                    "ndvi_min": float(valid_values.min()),
                    "ndvi_max": float(valid_values.max()),
                    "ndvi_mean": float(valid_values.mean())
                }
            )

        continue


    # -----------------------------------------------------
    # 5. Process new acquisition
    # -----------------------------------------------------

    print(
        f"Processing: "
        f"{acquisition_date:%Y-%m-%d} "
        f"{satellite}"
    )

    paths = date_rows["path"].tolist()


    # -----------------------------------------------------
    # 6. Open the two NDVI tiles
    # -----------------------------------------------------

    sources = [
        rasterio.open(path)
        for path in paths
    ]

    try:

        # Check that both tiles use the same CRS
        if sources[0].crs != sources[1].crs:

            raise ValueError(
                f"CRS mismatch for "
                f"{acquisition_date:%Y-%m-%d}"
            )


        # -------------------------------------------------
        # 7. Mosaic the two tiles
        # -------------------------------------------------

        mosaic, mosaic_transform = merge(
            sources,
            nodata=-9999.0,
            method="first"
        )

        mosaic_array = mosaic[0]


        # -------------------------------------------------
        # 8. Create temporary in-memory raster
        # -------------------------------------------------

        profile = sources[0].profile.copy()

        profile.update(
            driver="GTiff",
            height=mosaic_array.shape[0],
            width=mosaic_array.shape[1],
            transform=mosaic_transform,
            count=1,
            dtype="float32",
            nodata=-9999.0,
            compress="deflate"
        )


        with rasterio.io.MemoryFile() as memfile:

            with memfile.open(**profile) as temp:

                temp.write(
                    mosaic_array,
                    1
                )


                # -------------------------------------------------
                # 9. Crop mosaic to AOI
                # -------------------------------------------------

                ndvi_aoi, aoi_transform = mask(
                    temp,
                    aoi_geometry,
                    crop=True,
                    filled=True,
                    nodata=-9999.0
                )


        # -----------------------------------------------------
        # 10. Update output profile
        # -----------------------------------------------------

        aoi_profile = profile.copy()

        aoi_profile.update(
            height=ndvi_aoi.shape[1],
            width=ndvi_aoi.shape[2],
            transform=aoi_transform,
            count=1,
            dtype="float32",
            nodata=-9999.0,
            compress="deflate"
        )


        # -----------------------------------------------------
        # 11. Save AOI NDVI raster
        # -----------------------------------------------------

        with rasterio.open(
            output_path,
            "w",
            **aoi_profile
        ) as dst:

            dst.write(
                ndvi_aoi[0],
                1
            )

            dst.set_band_description(
                1,
                "NDVI"
            )


        # -----------------------------------------------------
        # 12. Basic statistics
        # -----------------------------------------------------

        values = ndvi_aoi[0]

        valid = values != -9999.0

        valid_values = values[valid]

        ndvi_aoi_records.append(
            {
                "acquisition_date": acquisition_date,
                "satellite": satellite,
                "path": str(output_path),
                "width": ndvi_aoi.shape[2],
                "height": ndvi_aoi.shape[1],
                "valid_pixels": int(valid.sum()),
                "ndvi_min": float(
                    valid_values.min()
                ),
                "ndvi_max": float(
                    valid_values.max()
                ),
                "ndvi_mean": float(
                    valid_values.mean()
                )
            }
        )

        print(
            f"  Saved: {output_path.name}"
        )

    finally:

        for source in sources:
            source.close()


# ---------------------------------------------------------
# 13. Create AOI NDVI inventory
# ---------------------------------------------------------

ndvi_aoi_inventory = pd.DataFrame(
    ndvi_aoi_records
)

print(
    f"\nAOI NDVI rasters available: "
    f"{len(ndvi_aoi_inventory)}"
)

display(
    ndvi_aoi_inventory
)

### 2.2 Validate AOI-Level NDVI Rasters

The final AOI-level NDVI rasters are validated to confirm that all expected acquisition dates were generated correctly and that the GeoTIFFs have consistent spatial and numerical properties.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio


# Directory containing the final AOI-level NDVI rasters
ndvi_aoi_dir = PROCESSED_SATELLITE_DIR / "ndvi_aoi"

ndvi_files = sorted(ndvi_aoi_dir.glob("*.tif"))

print(f"NDVI AOI rasters found: {len(ndvi_files)}")

validation_records = []

for path in ndvi_files:

    with rasterio.open(path) as src:

        data = src.read(1)

        valid_mask = (
            (data != src.nodata)
            & np.isfinite(data)
        )

        valid_values = data[valid_mask]

        if valid_values.size == 0:
            raise ValueError(f"No valid NDVI pixels found: {path.name}")

        validation_records.append({
            "file": path.name,
            "crs": src.crs.to_string(),
            "width": src.width,
            "height": src.height,
            "resolution_x": src.res[0],
            "resolution_y": src.res[1],
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "bands": src.count,
            "valid_pixels": valid_values.size,
            "min_ndvi": valid_values.min(),
            "max_ndvi": valid_values.max(),
            "mean_ndvi": valid_values.mean(),
        })


validation_df = pd.DataFrame(validation_records)

display(validation_df)


# ------------------------------------------------------------------
# Consistency checks
# ------------------------------------------------------------------

if not (validation_df["crs"] == "EPSG:32617").all():
    raise ValueError("Not all rasters use EPSG:32617.")

if not np.allclose(validation_df["resolution_x"], 10.0):
    raise ValueError("Not all rasters have a 10 m X resolution.")

if not np.allclose(validation_df["resolution_y"], 10.0):
    raise ValueError("Not all rasters have a 10 m Y resolution.")

if not (validation_df["dtype"] == "float32").all():
    raise ValueError("Not all rasters use float32.")

if not (validation_df["bands"] == 1).all():
    raise ValueError("Not all rasters contain exactly one band.")

if not (validation_df["nodata"] == -9999.0).all():
    raise ValueError("Not all rasters use -9999 as NoData.")

if (validation_df["valid_pixels"] <= 0).any():
    raise ValueError("At least one raster contains no valid pixels.")

if (validation_df["min_ndvi"] < -1).any():
    raise ValueError("NDVI values below -1 detected.")

if (validation_df["max_ndvi"] > 1).any():
    raise ValueError("NDVI values above 1 detected.")


print("✓ CRS: EPSG:32617")
print("✓ Resolution: 10 × 10 m")
print("✓ Data type: float32")
print("✓ Bands: 1")
print("✓ NoData: -9999")
print("✓ Valid NDVI pixels found in every raster")
print("✓ NDVI values within the expected [-1, 1] range")
print("\nFinal AOI NDVI raster validation completed successfully.")


## 3. Select Additional NDVI Candidates

The original NDVI sampling design contains 15 acquisition dates distributed across three periods:

* November–February: 4 dates
* March–April: 6 dates
* May–June: 5 dates

Eight dates have already been downloaded. This section selects seven additional AOI-screened candidates while excluding the completed dates and maintaining the planned period distribution.


In [ ]:

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Dates already available as complete AOI rasters
# ---------------------------------------------------------

completed_dates = pd.to_datetime([
    "2025-11-03",
    "2026-01-04",
    "2026-02-16",
    "2026-03-28",
    "2026-05-02",
    "2026-05-07",
    "2026-06-16",
    "2026-06-26",
])


# ---------------------------------------------------------
# 2. Target number of dates for each period
# ---------------------------------------------------------

target_counts = {
    "Nov-Feb": 4,
    "Mar-Apr": 6,
    "May-Jun": 5,
}


# ---------------------------------------------------------
# 3. Current number of completed dates
# ---------------------------------------------------------

completed_periods = pd.DataFrame({
    "acquisition_date": completed_dates
})

completed_periods["period"] = pd.cut(
    completed_periods["acquisition_date"],
    bins=pd.to_datetime([
        "2025-10-31",
        "2026-02-28",
        "2026-04-30",
        "2026-06-30",
    ]),
    labels=["Nov-Feb", "Mar-Apr", "May-Jun"],
)

current_counts = completed_periods["period"].value_counts().to_dict()

print("Current dates:")
for period, target in target_counts.items():
    current = current_counts.get(period, 0)
    print(f"{period}: {current} / {target}")


# ---------------------------------------------------------
# 4. Number of additional dates required
# ---------------------------------------------------------

additional_counts = {
    period: target_counts[period] - current_counts.get(period, 0)
    for period in target_counts
}

print("\nAdditional dates required:")
for period, count in additional_counts.items():
    print(f"{period}: {count}")


# ---------------------------------------------------------
# 5. Prepare the AOI-screened candidate pairs
# ---------------------------------------------------------

candidates = pair_screening.copy()

candidates["acquisition_date"] = pd.to_datetime(
    candidates["acquisition_date"]
)

# Exclude dates that are already complete
candidates = candidates[
    ~candidates["acquisition_date"].isin(completed_dates)
].copy()


# ---------------------------------------------------------
# 6. Keep only candidates suitable for NDVI analysis
# ---------------------------------------------------------

candidates = candidates[
    candidates["aoi_cloud_pct"] < 25
].copy()


# ---------------------------------------------------------
# 7. Select dates with good temporal spacing
# ---------------------------------------------------------

selected_additional = []

for period, number_needed in additional_counts.items():

    period_candidates = candidates[
        candidates["period"] == period
    ].copy()

    if len(period_candidates) < number_needed:
        raise ValueError(
            f"Only {len(period_candidates)} suitable candidates "
            f"available for {period}, but {number_needed} are required."
        )

    # Dates already represented in this period
    existing = completed_dates[
        (
            (completed_dates >= pd.Timestamp("2025-11-01"))
            & (completed_dates <= pd.Timestamp("2026-02-28"))
            & (period == "Nov-Feb")
        )
        |
        (
            (completed_dates >= pd.Timestamp("2026-03-01"))
            & (completed_dates <= pd.Timestamp("2026-04-30"))
            & (period == "Mar-Apr")
        )
        |
        (
            (completed_dates >= pd.Timestamp("2026-05-01"))
            & (completed_dates <= pd.Timestamp("2026-06-30"))
            & (period == "May-Jun")
        )
    ]

    chosen = []

    for _ in range(number_needed):

        remaining = period_candidates[
            ~period_candidates["acquisition_date"].isin(chosen)
        ].copy()

        # Dates already represented by either completed
        # or newly selected acquisitions
        represented = list(existing) + chosen

        if represented:
            remaining["distance_days"] = remaining[
                "acquisition_date"
            ].apply(
                lambda date: min(
                    abs((date - existing_date).days)
                    for existing_date in represented
                )
            )
        else:
            remaining["distance_days"] = np.inf

        # Prefer the date furthest from existing dates.
        # Lower AOI cloud breaks ties.
        remaining = remaining.sort_values(
            ["distance_days", "aoi_cloud_pct"],
            ascending=[False, True],
        )

        selected_date = remaining.iloc[0]["acquisition_date"]

        chosen.append(selected_date)

        selected_row = remaining.iloc[0].copy()
        selected_additional.append(selected_row)

        # Remove selected date from the candidate pool
        period_candidates = period_candidates[
            period_candidates["acquisition_date"] != selected_date
        ]


# ---------------------------------------------------------
# 8. Final additional selection
# ---------------------------------------------------------

ndvi_additional = pd.DataFrame(selected_additional)

ndvi_additional = ndvi_additional.sort_values(
    "acquisition_date"
).reset_index(drop=True)


print("\nAdditional candidate dates:")
display(
    ndvi_additional[
        [
            "acquisition_date",
            "satellite",
            "period",
            "aoi_cloud_pct",
        ]
    ]
)


# ---------------------------------------------------------
# 9. Verify the final 15-date distribution
# ---------------------------------------------------------

all_selected_dates = pd.concat(
    [
        pd.DataFrame({
            "acquisition_date": completed_dates
        }),
        ndvi_additional[
            ["acquisition_date"]
        ],
    ],
    ignore_index=True,
)

all_selected_dates["period"] = pd.cut(
    all_selected_dates["acquisition_date"],
    bins=pd.to_datetime([
        "2025-10-31",
        "2026-02-28",
        "2026-04-30",
        "2026-06-30",
    ]),
    labels=["Nov-Feb", "Mar-Apr", "May-Jun"],
)

print("\nFinal planned distribution:")
display(
    all_selected_dates["period"]
    .value_counts()
    .reindex(["Nov-Feb", "Mar-Apr", "May-Jun"])
)



## 4. Export Tables and Figures for Reporting

The available analytical DataFrames are exported as CSV files under `outputs/tables/` so they can be reused in the project report, PDF, or presentation. Any Matplotlib figures that are open when this cell runs are saved under `outputs/figures/`. This notebook does not currently create its own plots, so the figure directory may remain empty unless a figure has been created during the session.

Run this cell after the preceding workflow cells. The export is limited to DataFrames and figures that exist in the current notebook session.


In [ ]:
# ---------------------------------------------------------
# 1. Create report-output directories
# ---------------------------------------------------------

OUTPUTS_DIR = PROJECT_DIR / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# 2. Export available analytical tables
# ---------------------------------------------------------

table_names = [
    "ndvi_inventory",
    "ndvi_aoi_inventory",
    "validation_df",
    "completed_periods",
    "candidates",
    "selected_additional",
    "ndvi_additional",
    "pair_screening",
    "downloaded_bands",
    "ndvi_selection",
    "selected_acquisitions",
]

saved_tables = []

for table_name in table_names:
    table = globals().get(table_name)

    # Only DataFrames can be exported directly as CSV.
    if isinstance(table, pd.DataFrame) and not table.empty:
        output_path = TABLES_DIR / f"{table_name}.csv"
        table.to_csv(output_path, index=False)
        saved_tables.append(output_path.name)

print(f"Tables saved: {len(saved_tables)}")
for filename in saved_tables:
    print(f"  {filename}")


# ---------------------------------------------------------
# 3. Save figures created in this notebook session
# ---------------------------------------------------------

import matplotlib.pyplot as plt

figure_numbers = plt.get_fignums()

for figure_number in figure_numbers:
    figure = plt.figure(figure_number)
    output_path = FIGURES_DIR / f"ndvi_processing_figure_{figure_number:02d}.png"
    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

print(f"Figures saved: {len(figure_numbers)}")


# Workflow Summary

This notebook processes validated Sentinel-2 Level-2A imagery for the GeoSoil-Cuba Jaruco/Mayabeque area of interest (AOI). It calculates tile-level NDVI from complete B04/B08 pairs, mosaics and crops the resulting rasters to the AOI, validates the outputs, and selects additional candidate dates for the planned temporal analysis.

### Project setup and source data

- Set the project root and defined paths for raw Sentinel-2 imagery, processed satellite data, and the curated soil–vegetation GeoPackage.
- Used `soil_vegetation_analysis.gpkg` as the vector source for the AOI.
- Kept original Sentinel-2 JP2 assets separate from derived NDVI GeoTIFFs.

### NDVI calculation and AOI processing

- Calculated tile-level NDVI for complete B04 (red) and B08 (near-infrared) acquisition pairs.
- Reused existing tile-level NDVI outputs instead of regenerating them.
- Mosaicked the two Sentinel-2 tiles for each acquisition date and cropped each mosaic to the AOI.
- Stored tile-level and AOI-level NDVI rasters separately under `data/processed/satellite/`.

### Raster validation and candidate selection

- Checked that AOI-level rasters use EPSG:32617, 10 m resolution, `float32` data, one band, and `-9999` as NoData.
- Checked that valid NDVI values fall within the expected range of -1 to 1.
- Compared completed acquisition dates with the planned 15-date sampling design and selected additional AOI-screened candidates to improve temporal spacing.

### Workflow outputs

- Exported available analytical DataFrames as CSV files under `outputs/tables/`.
- Saved Matplotlib figures that were open in the current notebook session under `outputs/figures/`.
- The export includes only tables and figures available when the export cell runs; rerun that cell after updating processing or selection results.

### Stage 4 conclusions

This stage converts validated Sentinel-2 red and near-infrared inputs into NDVI rasters suitable for analysis over the project AOI. The tile-level products preserve the acquisition-specific processing results, while the AOI-level products provide a consistent spatial footprint for temporal comparisons.

Raster validation confirms key structural and numerical properties, but it does not establish that every valid pixel is free from cloud, haze, shadow, or other contamination. The SCL-based screening and the acquisition-selection criteria therefore remain important when interpreting the NDVI time series. Candidate dates selected here should be treated as a sampling plan until their imagery is available and validated. Regenerate the inventory and report tables after any changes to the input files or selected dates.
